In [24]:
#dataset loading

import pandas as pd
import numpy as np
from datetime import datetime
import calendar

df = pd.read_csv("Data set for DADS June.csv")

print("First Five Records")
display(df.head())

print("Dataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
display(df.dtypes)

print("\nMissing Values:")
display(df.isnull().sum())

print("\nSummary Statistics:")
display(df.describe(include='all'))

First Five Records


,Date,Time,Description,Type,Amount,Balance,Mode,Ref
0,2024-01-01,03:11,AMAZON SELLER SVCS,Debit,₹2462,678275.0,UPI,TXN190872
1,01-Jan-24,05:44,BHIM-BMTC,DR,50.00,681007.0,UPI,TXN143064
2,01-Jan-24,09:35,NEFT-TECHCRUSH LABS-SALARY MAY24,CR,₹84728,484728.0,NEFT,TXN246316
3,2024-01-01,14:07,UPI-AMAN-8934@OKAXIS,Debit,₹1828,-748745.0,UPI,TXN569226
4,01 Jan 2024,14:23,BHIM-BLINKIT,Debit,270.00,680737.0,UPI,TXN968962


Dataset Shape:
(1328, 8)

Column Names:
['Date', 'Time', 'Description', 'Type', 'Amount', 'Balance', 'Mode', 'Ref']

Data Types:


,0
Date,object
Time,object
Description,object
Type,object
Amount,object
Balance,float64
Mode,object
Ref,object



Missing Values:


,0
Date,0
Time,0
Description,0
Type,0
Amount,0
Balance,0
Mode,0
Ref,0



Summary Statistics:


,Date,Time,Description,Type,Amount,Balance,Mode,Ref
count,1328,1328,1328,1328,1328,1328.000000,1328,1328
unique,617,832,283,3,1141,NaN,4,1307
top,2024-02-07,16:19,POS SWIGGY BANGALORE,DR,"Rs. 15,000",NaN,UPI,TXN427270
freq,6,6,30,670,5,NaN,1299,2
mean,NaN,NaN,NaN,NaN,NaN,147805.330572,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,341059.809969,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,-769127.000000,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,-104585.750000,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,151650.000000,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,448324.250000,NaN,NaN


In [3]:
#Feature 1: Parse Dates and Basic Data Cleaning

clean_df = df.copy()
clean_df["Date"] = pd.to_datetime(
    clean_df["Date"],
    errors="coerce",
    format="mixed"
)

text_columns = ["Description", "Type", "Mode", "Ref"]
for column in text_columns:
    clean_df[column] = clean_df[column].astype(str).str.strip()
clean_df["Amount"] = pd.to_numeric(clean_df["Amount"], errors="coerce")
clean_df["Balance"] = pd.to_numeric(clean_df["Balance"], errors="coerce")
print("Missing Values Before Cleaning")
display(clean_df.isnull().sum())

clean_df = clean_df.dropna(subset=["Date", "Amount"])
duplicates = clean_df.duplicated().sum()
print("Duplicate Rows Found :", duplicates)

clean_df = clean_df.drop_duplicates()
clean_df["Year"] = clean_df["Date"].dt.year
clean_df["Month"] = clean_df["Date"].dt.month
clean_df["Month_Name"] = clean_df["Date"].dt.month_name()
clean_df["Day"] = clean_df["Date"].dt.day
clean_df["Day_Name"] = clean_df["Date"].dt.day_name()

print("\nDataset Shape After Cleaning:")
print(clean_df.shape)

print("\nFirst Five Cleaned Records")
display(clean_df.head())

print("\nData Types")
display(clean_df.dtypes)

Missing Values Before Cleaning


,0
Date,0
Time,0
Description,0
Type,0
Amount,813
Balance,0
Mode,0
Ref,0


Duplicate Rows Found : 6

Dataset Shape After Cleaning:
(509, 13)

First Five Cleaned Records


,Date,Time,Description,Type,Amount,Balance,Mode,Ref,Year,Month,Month_Name,Day,Day_Name
1,2024-01-01,05:44,BHIM-BMTC,DR,50.0,681007.0,UPI,TXN143064,2024,1,January,1,Monday
4,2024-01-01,14:23,BHIM-BLINKIT,Debit,270.0,680737.0,UPI,TXN968962,2024,1,January,1,Monday
9,2024-02-01,06:55,UPI-GROWWPAY@HDFCBANK,DR,3956.0,672221.0,UPI,TXN418594,2024,2,February,1,Thursday
12,2024-02-01,11:29,POS OLA-PRIME,DR,258.0,676762.0,UPI,TXN389930,2024,2,February,1,Thursday
14,2024-01-02,14:01,UPI-STARBUCKS@AXIS,Debit,266.0,670051.0,UPI,TXN283700,2024,1,January,2,Tuesday



Data Types


,0
Date,datetime64[ns]
Time,object
Description,object
Type,object
Amount,float64
Balance,float64
Mode,object
Ref,object
Year,int32
Month,int32


In [4]:
#FEATURE 2: VENDOR EXTRACTION (WITHOUT REGEX)

def extract_vendor(description):
    if pd.isna(description):
        return "Unknown"
    description = str(description).upper().strip()
    parts = description.split("/")
    ignore_words = [
        "UPI", "IMPS", "NEFT", "RTGS", "ATM", "POS",
        "BANK", "PAYMENT", "TRANSFER", "TO", "FROM",
        "BY", "DR", "CR", "REF", "MB", "UPI-P2A"
    ]
    for part in parts:
        part = part.strip()
        if len(part) < 3:
            continue
        if part in ignore_words:
            continue
        vendor = ""
        for ch in part:
            if not ch.isdigit():
                vendor += ch
        vendor = vendor.replace("-", " ")
        vendor = vendor.replace("_", " ")
        vendor = vendor.strip()
        if len(vendor) > 2:
            return vendor.title()
    return "Unknown"
clean_df["Vendor"] = clean_df["Description"].apply(extract_vendor)
print("=" * 60)
print("Sample Vendor Extraction")
print("=" * 60)
display(clean_df[["Description", "Vendor"]].head(20))
vendor_count = (
    clean_df["Vendor"]
    .value_counts()
    .reset_index()
)
vendor_count.columns = ["Vendor", "Transactions"]

print("\nTop 15 Vendors")
display(vendor_count.head(15))

print("\nTotal Unique Vendors :", clean_df["Vendor"].nunique())

print("\nInterpretation:")
print("- Vendor names were extracted using only string operations.")
print("- Regular expressions were not used.")
print("- Banking keywords and numeric values were ignored.")
print("- The extracted Vendor column will be used for further analysis.")

Sample Vendor Extraction


,Description,Vendor
1,BHIM-BMTC,Bhim Bmtc
4,BHIM-BLINKIT,Bhim Blinkit
9,UPI-GROWWPAY@HDFCBANK,Upi Growwpay@Hdfcbank
12,POS OLA-PRIME,Pos Ola Prime
14,UPI-STARBUCKS@AXIS,Upi Starbucks@Axis
16,ANI Technologies,Ani Technologies
21,GROFERS INDIA P L,Grofers India P L
24,BANGALORE ELEC SUPPLY,Bangalore Elec Supply
30,OLA CABS,Ola Cabs
32,UPI-AMAZONPAY@HDFCBANK,Upi Amazonpay@Hdfcbank



Top 15 Vendors


,Vendor,Transactions
0,Upi Rapido@Okaxis,17
1,Swiggy Instamart,16
2,Pos Swiggy Bangalore,14
3,Zomato Media P L,13
4,Pos Zepto Bangalore,12
5,Uber India,12
6,Upi Swiggy@Okaxis,12
7,Ola Electric,11
8,Ola Cabs,11
9,Upi Starbucks@Axis,11



Total Unique Vendors : 113

Interpretation:
- Vendor names were extracted using only string operations.
- Regular expressions were not used.
- Banking keywords and numeric values were ignored.
- The extracted Vendor column will be used for further analysis.


In [5]:
#Feature 3: automatic category tagging

def assign_category(description):
    if pd.isna(description):
        return "Others"
    description = str(description).lower()

    if ("swiggy" in description or
        "zomato" in description or
        "restaurant" in description or
        "cafe" in description or
        "hotel" in description or
        "food" in description or
        "dominos" in description or
        "pizza" in description or
        "kfc" in description or
        "burger" in description):
        return "Food"

    elif ("dmart" in description or
          "mart" in description or
          "supermarket" in description or
          "grocery" in description or
          "reliance fresh" in description or
          "more" in description or
          "bigbasket" in description):
        return "Groceries"

    elif ("amazon" in description or
          "flipkart" in description or
          "myntra" in description or
          "ajio" in description or
          "shopping" in description):
        return "Shopping"

    elif ("petrol" in description or
          "fuel" in description or
          "hpcl" in description or
          "iocl" in description or
          "indianoil" in description or
          "bharat petroleum" in description):
        return "Fuel"

    elif ("uber" in description or
          "ola" in description or
          "irctc" in description or
          "metro" in description or
          "bus" in description or
          "flight" in description or
          "makemytrip" in description):
        return "Travel"

    elif ("netflix" in description or
          "prime" in description or
          "spotify" in description or
          "hotstar" in description or
          "movie" in description or
          "bookmyshow" in description):
        return "Entertainment"

    elif ("hospital" in description or
          "medical" in description or
          "clinic" in description or
          "pharmacy" in description or
          "apollo" in description or
          "medicine" in description):
        return "Healthcare"

    elif ("electricity" in description or
          "water" in description or
          "gas" in description or
          "recharge" in description or
          "broadband" in description or
          "wifi" in description or
          "airtel" in description or
          "jio" in description or
          "bsnl" in description):
        return "Bills"

    elif ("college" in description or
          "school" in description or
          "university" in description or
          "course" in description or
          "udemy" in description or
          "coursera" in description):
        return "Education"

    elif ("mutual" in description or
          "sip" in description or
          "stock" in description or
          "zerodha" in description or
          "groww" in description or
          "investment" in description):
        return "Investment"

    elif ("upi" in description or
          "imps" in description or
          "neft" in description or
          "rtgs" in description or
          "transfer" in description):
        return "Transfer"

    else:
        return "Others"

clean_df["Category"] = clean_df["Description"].apply(assign_category)
print("=" * 70)
print("Sample Category Assignment")
print("=" * 70)
display(clean_df[["Description", "Category"]].head(20))
category_count = (
    clean_df["Category"]
    .value_counts()
    .reset_index()
)
category_count.columns = ["Category", "Transactions"]
print("\nTransaction Count by Category")
display(category_count)
category_spending = (
    clean_df.groupby("Category")["Amount"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
category_spending.columns = ["Category", "Total Spending"]
print("\nTotal Spending by Category")
display(category_spending)
print("\nInterpretation:")
print("- Transactions were automatically classified into predefined categories.")
print("- Categorization was performed using only Python string matching.")
print("- No Machine Learning, Regular Expressions, or external libraries were used.")
print("- These categories will be used for spending analysis and dashboard generation.")

Sample Category Assignment


,Description,Category
1,BHIM-BMTC,Others
4,BHIM-BLINKIT,Others
9,UPI-GROWWPAY@HDFCBANK,Investment
12,POS OLA-PRIME,Travel
14,UPI-STARBUCKS@AXIS,Transfer
16,ANI Technologies,Others
21,GROFERS INDIA P L,Others
24,BANGALORE ELEC SUPPLY,Others
30,OLA CABS,Travel
32,UPI-AMAZONPAY@HDFCBANK,Shopping



Transaction Count by Category


,Category,Transactions
0,Food,148
1,Others,143
2,Travel,77
3,Transfer,58
4,Shopping,38
5,Groceries,19
6,Entertainment,12
7,Investment,6
8,Bills,6
9,Fuel,2



Total Spending by Category


,Category,Total Spending
0,Shopping,178040.0
1,Transfer,129195.0
2,Investment,78956.0
3,Food,71534.0
4,Others,70119.0
5,Travel,22493.0
6,Groceries,13415.0
7,Entertainment,5572.0
8,Bills,3336.0
9,Fuel,1714.0



Interpretation:
- Transactions were automatically classified into predefined categories.
- Categorization was performed using only Python string matching.
- No Machine Learning, Regular Expressions, or external libraries were used.
- These categories will be used for spending analysis and dashboard generation.


In [7]:
import pandas as pd

# FEATURE 4 : SPENDING OVERVIEW

print("=" * 70)
print("             SPENDING OVERVIEW")
print("=" * 70)

income_df = clean_df[clean_df["Type"].str.lower() == "credit"]
expense_df = clean_df[clean_df["Type"].str.lower() == "debit"]
total_income = income_df["Amount"].sum()
total_expense = expense_df["Amount"].sum()
net_balance = total_income - total_expense
average_transaction = clean_df["Amount"].mean()
highest_transaction = clean_df["Amount"].max()
lowest_transaction = clean_df["Amount"].min()
total_transactions = len(clean_df)

print(f"Total Transactions      : {total_transactions}")
print(f"Total Income            : ₹ {total_income:.2f}")
print(f"Total Expense           : ₹ {total_expense:.2f}")
print(f"Net Savings             : ₹ {net_balance:.2f}")
print(f"Average Transaction     : ₹ {average_transaction:.2f}")
print(f"Highest Transaction     : ₹ {highest_transaction:.2f}")
print(f"Lowest Transaction     : ₹ {lowest_transaction:.2f}")
print("\nLargest Expense Transaction")

# Check if expense_df is not empty before finding idxmax
if not expense_df.empty:
    largest_expense = expense_df.loc[
        expense_df["Amount"].idxmax()
    ]
    display(largest_expense)
else:
    print("No expense transactions found.")

print("\nLargest Income Transaction")
# Check if income_df is not empty before finding idxmax
if not income_df.empty:
    largest_income = income_df.loc[
        income_df["Amount"].idxmax()
    ]
    display(largest_income)
else:
    print("No income transactions found.")

print("\nCategory-wise Spending")
category_summary = (
    expense_df.groupby("Category")["Amount"]
    .agg(["count", "sum", "mean"])
    .reset_index()
)
category_summary.columns = [
    "Category",
    "Transactions",
    "Total Spending",
    "Average Spending"
]
category_summary = category_summary.sort_values(
    by="Total Spending",
    ascending=False
)
display(category_summary)

print("\nTop Vendors by Spending")
vendor_summary = (
    expense_df.groupby("Vendor")["Amount"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
vendor_summary.columns = [
    "Vendor",
    "Total Spending"
]
display(vendor_summary.head(10))

print("\n" + "=" * 70)
print("INTERPRETATION")
print("=" * 70)

print(f"• Total Income recorded is ₹{total_income:.2f}.")
print(f"• Total Expenses recorded are ₹{total_expense:.2f}.")
print(f"• Net Savings are ₹{net_balance:.2f}.")
print(f"• Average transaction value is ₹{average_transaction:.2f}.")
print("• Category-wise analysis identifies where most money is spent.")
print("• Vendor-wise analysis highlights the merchants receiving the highest payments.")
print("• These insights help understand overall financial behaviour and spending habits.")

             SPENDING OVERVIEW
Total Transactions      : 509
Total Income            : ₹ 0.00
Total Expense           : ₹ 234628.00
Net Savings             : ₹ -234628.00
Average Transaction     : ₹ 1128.44
Highest Transaction     : ₹ 85393.00
Lowest Transaction     : ₹ 10.00

Largest Expense Transaction


,450
Date,2024-03-02 00:00:00
Time,11:58
Description,AMAZON IN
Type,Debit
Amount,18273.0
Balance,-558650.0
Mode,UPI
Ref,TXN228115
Year,2024
Month,3



Largest Income Transaction
No income transactions found.

Category-wise Spending


,Category,Transactions,Total Spending,Average Spending
6,Shopping,17,71928.0,4231.058824
5,Others,78,46330.0,593.974359
4,Investment,3,45000.0,15000.000000
2,Food,74,34898.0,471.594595
7,Transfer,29,14315.0,493.620690
8,Travel,40,12105.0,302.625000
3,Groceries,10,6114.0,611.400000
0,Bills,4,2286.0,571.500000
1,Entertainment,6,1652.0,275.333333



Top Vendors by Spending


,Vendor,Total Spending
0,Imps Zerodha Coin,30000.0
1,Amazon In,18273.0
2,Upi Zerodha Coin@Axis,15000.0
3,Upi Flipkart@Hdfcbank,14416.0
4,Myntra Designs,11391.0
5,Pos Flipkart Bangalore,8534.0
6,Amazonin Marketplace,5430.0
7,Amazon Seller Svcs,5001.0
8,Atm Wdl Icici,5000.0
9,Upi Amazonpay@Hdfcbank,4830.0



INTERPRETATION
• Total Income recorded is ₹0.00.
• Total Expenses recorded are ₹234628.00.
• Net Savings are ₹-234628.00.
• Average transaction value is ₹1128.44.
• Category-wise analysis identifies where most money is spent.
• Vendor-wise analysis highlights the merchants receiving the highest payments.
• These insights help understand overall financial behaviour and spending habits.


In [8]:
# FEATURE 5 : MONTHLY SPENDING TRENDS

print("=" * 70)
print("            FEATURE 5 : MONTHLY SPENDING TRENDS")
print("=" * 70)

income_df = clean_df[clean_df["Type"].str.lower() == "credit"]
expense_df = clean_df[clean_df["Type"].str.lower() == "debit"]
monthly_income = (
    income_df.groupby(["Year", "Month_Name"])["Amount"]
    .sum()
    .reset_index()
)
monthly_income.columns = ["Year", "Month", "Total Income"]

monthly_expense = (
    expense_df.groupby(["Year", "Month_Name"])["Amount"]
    .sum()
    .reset_index()
)
monthly_expense.columns = ["Year", "Month", "Total Expense"]

monthly_summary = pd.merge(
    monthly_income,
    monthly_expense,
    on=["Year", "Month"],
    how="outer"
)
monthly_summary = monthly_summary.fillna(0)
monthly_summary["Savings"] = (
    monthly_summary["Total Income"] -
    monthly_summary["Total Expense"]
)

print("\nMonthly Spending Summary\n")
display(monthly_summary)
average_monthly_spending = monthly_summary["Total Expense"].mean()
print(f"\nAverage Monthly Spending : ₹ {average_monthly_spending:.2f}")

highest_month = monthly_summary.loc[
    monthly_summary["Total Expense"].idxmax()
]
print("\nHighest Spending Month")
display(highest_month)
lowest_month = monthly_summary.loc[
    monthly_summary["Total Expense"].idxmin()
]
print("\nLowest Spending Month")
display(lowest_month)

monthly_transactions = (
    clean_df.groupby(["Year", "Month_Name"])
    .size()
    .reset_index(name="Transactions")
)
print("\nMonthly Transaction Count\n")
display(monthly_transactions)

monthly_category = (
    expense_df.groupby(["Month_Name", "Category"])["Amount"]
    .sum()
    .reset_index()
)
print("\nMonthly Category-wise Spending\n")
display(monthly_category)

print("\n" + "=" * 70)
print("INTERPRETATION")
print("=" * 70)
print(f"• Average monthly spending is ₹{average_monthly_spending:.2f}.")
print(f"• The highest spending occurred in {highest_month['Month']}.")
print(f"• The lowest spending occurred in {lowest_month['Month']}.")
print("• Monthly income, expenses, and savings provide an overview of financial performance.")
print("• Monthly category analysis helps identify which categories contribute the most to expenses.")
print("• These insights can be used for budgeting and future financial planning.")

            FEATURE 5 : MONTHLY SPENDING TRENDS

Monthly Spending Summary



,Year,Month,Total Income,Total Expense,Savings
0,2024,April,0.0,34345.0,-34345.0
1,2024,August,0.0,640.0,-640.0
2,2024,December,0.0,485.0,-485.0
3,2024,February,0.0,21480.0,-21480.0
4,2024,January,0.0,19296.0,-19296.0
5,2024,July,0.0,886.0,-886.0
6,2024,June,0.0,21979.0,-21979.0
7,2024,March,0.0,38299.0,-38299.0
8,2024,May,0.0,88051.0,-88051.0
9,2024,October,0.0,5503.0,-5503.0



Average Monthly Spending : ₹ 21329.82

Highest Spending Month


,8
Year,2024
Month,May
Total Income,0.0
Total Expense,88051.0
Savings,-88051.0



Lowest Spending Month


...


Monthly Transaction Count



,Year,Month_Name,Transactions
0,2024,April,86
1,2024,August,3
2,2024,December,3
3,2024,February,79
4,2024,January,70
5,2024,July,5
6,2024,June,72
7,2024,March,85
8,2024,May,94
9,2024,November,2



Monthly Category-wise Spending



,Month_Name,Category,Amount
0,April,Bills,708.0
1,April,Entertainment,451.0
2,April,Food,4308.0
3,April,Groceries,1926.0
4,April,Investment,15000.0
5,April,Others,3606.0
6,April,Shopping,1398.0
7,April,Transfer,5433.0
8,April,Travel,1515.0
9,August,Others,640.0



INTERPRETATION
• Average monthly spending is ₹21329.82.
• The highest spending occurred in May.
• The lowest spending occurred in December.
• Monthly income, expenses, and savings provide an overview of financial performance.
• Monthly category analysis helps identify which categories contribute the most to expenses.
• These insights can be used for budgeting and future financial planning.


In [9]:
# FEATURE 6 : TIME-OF-DAY SPENDING ANALYSIS

print("=" * 70)
print("           FEATURE 6 : TIME-OF-DAY SPENDING ANALYSIS")
print("=" * 70)

clean_df["Hour"] = clean_df["Date"].dt.hour
def get_time_slot(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

clean_df["Time_Slot"] = clean_df["Hour"].apply(get_time_slot)
expense_df = clean_df[clean_df["Type"].str.lower() == "debit"]
time_summary = (
    expense_df.groupby("Time_Slot")["Amount"]
    .agg(["count", "sum", "mean"])
    .reset_index()
)
time_summary.columns = [
    "Time Slot",
    "Transactions",
    "Total Spending",
    "Average Spending"
]
print("\nSpending by Time Slot\n")
display(time_summary)

highest_slot = time_summary.loc[
    time_summary["Total Spending"].idxmax()
]
print("\nHighest Spending Time Slot\n")
display(highest_slot)
vendor_time = (
    expense_df.groupby(["Time_Slot", "Vendor"])["Amount"]
    .sum()
    .reset_index()
)
vendor_time = vendor_time.sort_values(
    by="Amount",
    ascending=False
)
print("\nTop Vendor Spending by Time Slot\n")
display(vendor_time.head(15))

category_time = (
    expense_df.groupby(["Time_Slot", "Category"])["Amount"]
    .sum()
    .reset_index()
)
print("\nCategory-wise Spending by Time Slot\n")
display(category_time)
transaction_count = (
    expense_df.groupby("Time_Slot")
    .size()
    .reset_index(name="Transaction Count")
)
print("\nTransaction Count by Time Slot\n")
display(transaction_count)

print("\n" + "=" * 70)
print("INTERPRETATION")
print("=" * 70)
print(f"• The highest spending occurred during the {highest_slot['Time Slot']} period.")
print("• The analysis divides transactions into Morning, Afternoon, Evening, and Night.")
print("• Vendor-wise analysis identifies the merchants most frequently used during each time period.")
print("• Category-wise analysis highlights the type of expenses made at different times of the day.")
print("• These insights help understand daily spending behaviour and financial habits.")

           FEATURE 6 : TIME-OF-DAY SPENDING ANALYSIS

Spending by Time Slot



,Time Slot,Transactions,Total Spending,Average Spending
0,Night,261,234628.0,898.957854



Highest Spending Time Slot



,0
Time Slot,Night
Transactions,261
Total Spending,234628.0
Average Spending,898.957854



Top Vendor Spending by Time Slot



...


Category-wise Spending by Time Slot



,Time_Slot,Category,Amount
0,Night,Bills,2286.0
1,Night,Entertainment,1652.0
2,Night,Food,34898.0
3,Night,Groceries,6114.0
4,Night,Investment,45000.0
5,Night,Others,46330.0
6,Night,Shopping,71928.0
7,Night,Transfer,14315.0
8,Night,Travel,12105.0



Transaction Count by Time Slot



,Time_Slot,Transaction Count
0,Night,261



INTERPRETATION
• The highest spending occurred during the Night period.
• The analysis divides transactions into Morning, Afternoon, Evening, and Night.
• Vendor-wise analysis identifies the merchants most frequently used during each time period.
• Category-wise analysis highlights the type of expenses made at different times of the day.
• These insights help understand daily spending behaviour and financial habits.


In [10]:
# FEATURE 7 : CATEGORY-WISE ANOMALY DETECTION

print("=" * 70)
print("          FEATURE 7 : CATEGORY-WISE ANOMALY DETECTION")
print("=" * 70)

expense_df = clean_df[clean_df["Type"].str.lower() == "debit"].copy()
expense_df["Z_Score"] = 0.0
categories = expense_df["Category"].unique()
for category in categories:
    category_data = expense_df[expense_df["Category"] == category]
    mean_amount = category_data["Amount"].mean()
    std_amount = category_data["Amount"].std()
    if pd.isna(std_amount) or std_amount == 0:
        continue
    z_scores = (category_data["Amount"] - mean_amount) / std_amount
    expense_df.loc[category_data.index, "Z_Score"] = z_scores
anomalies = expense_df[
    expense_df["Z_Score"].abs() > 2
].copy()

print("\nDetected Anomalous Transactions\n")
if len(anomalies) == 0:
    print("No anomalous transactions were detected.")
else:
    display(
        anomalies[
            [
                "Date",
                "Vendor",
                "Category",
                "Amount",
                "Z_Score"
            ]
        ].sort_values(by="Z_Score", ascending=False)
    )
print("\nNumber of Anomalies in Each Category\n")

anomaly_summary = (
    anomalies.groupby("Category")
    .size()
    .reset_index(name="Anomaly Count")
)
if len(anomaly_summary) == 0:
    print("No anomalies found in any category.")
else:
    display(anomaly_summary)
if len(anomalies) > 0:
    print("\nHighest Anomalous Transaction\n")
    highest_anomaly = anomalies.loc[
        anomalies["Amount"].idxmax()
    ]
    display(highest_anomaly)
total_transactions = len(expense_df)
total_anomalies = len(anomalies)
percentage = (total_anomalies / total_transactions) * 100
print(f"\nTotal Expense Transactions : {total_transactions}")
print(f"Total Anomalies Detected   : {total_anomalies}")
print(f"Percentage of Anomalies    : {percentage:.2f}%")

print("\n" + "=" * 70)
print("INTERPRETATION")
print("=" * 70)
print("• Anomalies were detected using the manual Z-score method.")
print("• The mean and standard deviation were calculated separately for each spending category.")
print("• Transactions having |Z-score| greater than 2 were considered unusual.")
print("• These transactions represent spending behaviour that significantly differs from the user's normal spending pattern.")
print("• This analysis helps identify unexpectedly large expenses that may require further investigation.")

          FEATURE 7 : CATEGORY-WISE ANOMALY DETECTION

Detected Anomalous Transactions



,Date,Vendor,Category,Amount,Z_Score
1180,2024-10-06,Atm Wdl Icici,Others,5000.0,6.423671
1284,2024-06-24,Upi Restaurant @Paytm,Food,1610.0,4.212895
1038,2024-05-22,Pos Empire Restaurant,Food,1549.0,3.987153
785,2024-04-17,Upi Bescom Bill@Hdfcbank,Transfer,2394.0,3.849554
1133,2024-06-04,Amzn Intpymt,Others,3066.0,3.604037
627,2024-03-27,Upi Restaurant @Paytm,Food,1345.0,3.232210
213,2024-01-31,Avenue Supermarts,Groceries,1921.0,2.687495
450,2024-03-02,Amazon In,Shopping,18273.0,2.647820
1032,2024-05-21,Pos Meghana Foods,Food,1171.0,2.588289
954,2024-05-11,Bmtc Bus Pass,Travel,43.0,-2.192242



Number of Anomalies in Each Category



,Category,Anomaly Count
0,Food,4
1,Groceries,1
2,Others,2
3,Shopping,1
4,Transfer,1
5,Travel,3



Highest Anomalous Transaction



,450
Date,2024-03-02 00:00:00
Time,11:58
Description,AMAZON IN
Type,Debit
Amount,18273.0
Balance,-558650.0
Mode,UPI
Ref,TXN228115
Year,2024
Month,3



Total Expense Transactions : 261
Total Anomalies Detected   : 12
Percentage of Anomalies    : 4.60%

INTERPRETATION
• Anomalies were detected using the manual Z-score method.
• The mean and standard deviation were calculated separately for each spending category.
• Transactions having |Z-score| greater than 2 were considered unusual.
• These transactions represent spending behaviour that significantly differs from the user's normal spending pattern.
• This analysis helps identify unexpectedly large expenses that may require further investigation.


In [13]:
# FEATURE 8 : SPENDING ARCHETYPES

print("=" * 70)
print("              FEATURE 8 : SPENDING ARCHETYPES")
print("=" * 70)

expense_df = clean_df[clean_df["Type"].str.lower() == "debit"].copy()
total_expense = expense_df["Amount"].sum()
average_expense = expense_df["Amount"].mean()
category_spending = (
    expense_df.groupby("Category")["Amount"]
    .sum()
)
vendor_spending = (
    expense_df.groupby("Vendor")["Amount"]
    .sum()
)

def foodie():
    food = category_spending.get("Food", 0)
    return food > 0.30 * total_expense
def shopaholic():
    shopping = category_spending.get("Shopping", 0)
    return shopping > 0.30 * total_expense
def traveller():
    travel = category_spending.get("Travel", 0)
    return travel > 0.20 * total_expense
def entertainment_lover():
    entertainment = category_spending.get("Entertainment", 0)
    return entertainment > 0.20 * total_expense
def health_conscious():
    health = category_spending.get("Healthcare", 0)
    return health > 0.15 * total_expense
def investor():
    invest = category_spending.get("Investment", 0)
    return invest > 0.20 * total_expense
def bill_payer():
    bills = category_spending.get("Bills", 0)
    return bills > 0.25 * total_expense
def saver():
    return average_expense < expense_df["Amount"].median()

archetypes = []
if foodie():
    archetypes.append("Foodie")
if shopaholic():
    archetypes.append("Shopaholic")
if traveller():
    archetypes.append("Traveller")
if entertainment_lover():
    archetypes.append("Entertainment Lover")
if health_conscious():
    archetypes.append("Health Conscious")
if investor():
    archetypes.append("Investor")
if bill_payer():
    archetypes.append("Bill Payer")
if saver():
    archetypes.append("Saver")
print("\nDetected Spending Archetypes\n")
if len(archetypes) == 0:
    print("No dominant spending archetype detected.")
else:
    for i, archetype in enumerate(archetypes, start=1):
        print(f"{i}. {archetype}")

print("\nCategory-wise Spending\n")
category_table = (
    expense_df.groupby("Category")["Amount"]
    .agg(["count", "sum", "mean"])
    .reset_index()
)
category_table.columns = [
    "Category",
    "Transactions",
    "Total Spending",
    "Average Spending"
]
category_table = category_table.sort_values(
    by="Total Spending",
    ascending=False
)
display(category_table)

print("\nTop Vendors by Spending\n")
top_vendors = (
    expense_df.groupby("Vendor")["Amount"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
top_vendors.columns = ["Vendor", "Total Spending"]
display(top_vendors)

print("\nCategory-wise Spending Percentage\n")
percentage_table = category_table.copy()
percentage_table["Percentage"] = (
    percentage_table["Total Spending"] /
    total_expense
) * 100
display(percentage_table)

print("\n" + "=" * 70)
print("INTERPRETATION")
print("=" * 70)
if len(archetypes) == 0:
    print("The user's spending is balanced across multiple categories.")
else:
    print("Detected Spending Personality:")
    for archetype in archetypes:
        print(f"• {archetype}")
print("\nKey Observations:")
print("• Spending behaviour was analysed using predefined rule-based conditions.")
print("• No Machine Learning or external libraries were used.")
print("• The archetypes represent dominant spending habits based on category-wise expenditure.")
print("• These insights can help users understand and improve their financial behaviour.")

              FEATURE 8 : SPENDING ARCHETYPES

Detected Spending Archetypes

1. Shopaholic

Category-wise Spending



,Category,Transactions,Total Spending,Average Spending
6,Shopping,17,71928.0,4231.058824
5,Others,78,46330.0,593.974359
4,Investment,3,45000.0,15000.000000
2,Food,74,34898.0,471.594595
7,Transfer,29,14315.0,493.620690
8,Travel,40,12105.0,302.625000
3,Groceries,10,6114.0,611.400000
0,Bills,4,2286.0,571.500000
1,Entertainment,6,1652.0,275.333333



Top Vendors by Spending



,Vendor,Total Spending
0,Imps Zerodha Coin,30000.0
1,Amazon In,18273.0
2,Upi Zerodha Coin@Axis,15000.0
3,Upi Flipkart@Hdfcbank,14416.0
4,Myntra Designs,11391.0
5,Pos Flipkart Bangalore,8534.0
6,Amazonin Marketplace,5430.0
7,Amazon Seller Svcs,5001.0
8,Atm Wdl Icici,5000.0
9,Upi Amazonpay@Hdfcbank,4830.0



Category-wise Spending Percentage



,Category,Transactions,Total Spending,Average Spending,Percentage
6,Shopping,17,71928.0,4231.058824,30.656188
5,Others,78,46330.0,593.974359,19.746151
4,Investment,3,45000.0,15000.000000,19.179297
2,Food,74,34898.0,471.594595,14.873758
7,Transfer,29,14315.0,493.620690,6.101147
8,Travel,40,12105.0,302.625000,5.159231
3,Groceries,10,6114.0,611.400000,2.605827
0,Bills,4,2286.0,571.500000,0.974308
1,Entertainment,6,1652.0,275.333333,0.704093



INTERPRETATION
Detected Spending Personality:
• Shopaholic

Key Observations:
• Spending behaviour was analysed using predefined rule-based conditions.
• No Machine Learning or external libraries were used.
• The archetypes represent dominant spending habits based on category-wise expenditure.
• These insights can help users understand and improve their financial behaviour.


In [14]:
# BONUS FEATURE 1 : DAY-OF-WEEK SPENDING ANALYSIS

print("=" * 70)
print("        BONUS FEATURE 1 : DAY-OF-WEEK SPENDING ANALYSIS")
print("=" * 70)

expense_df = clean_df[clean_df["Type"].str.lower() == "debit"].copy()
day_summary = (
    expense_df.groupby("Day_Name")["Amount"]
    .agg(["count", "sum", "mean"])
    .reset_index()
)
day_summary.columns = [
    "Day",
    "Transactions",
    "Total Spending",
    "Average Spending"
]
day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]
day_summary["Day"] = pd.Categorical(
    day_summary["Day"],
    categories=day_order,
    ordered=True
)
day_summary = day_summary.sort_values("Day")
print("\nDay-wise Spending Summary\n")
display(day_summary)
highest_day = day_summary.loc[
    day_summary["Total Spending"].idxmax()
]
print("\nHighest Spending Day\n")
display(highest_day)

lowest_day = day_summary.loc[
    day_summary["Total Spending"].idxmin()
]
print("\nLowest Spending Day\n")
display(lowest_day)
vendor_day = (
    expense_df.groupby(["Day_Name", "Vendor"])["Amount"]
    .sum()
    .reset_index()
)
vendor_day = vendor_day.sort_values(
    by="Amount",
    ascending=False
)
print("\nTop Vendor Spending by Day\n")
display(vendor_day.head(15))
category_day = (
    expense_df.groupby(["Day_Name", "Category"])["Amount"]
    .sum()
    .reset_index()
)
print("\nCategory-wise Spending by Day\n")
display(category_day)

print("\n" + "=" * 70)
print("INTERPRETATION")
print("=" * 70)
print(f"• The highest spending occurred on {highest_day['Day']}.")
print(f"• The lowest spending occurred on {lowest_day['Day']}.")
print("• Day-wise analysis helps identify weekly spending patterns.")
print("• Vendor-wise and category-wise summaries reveal where money is spent most on different days.")
print("• These insights support better weekly budgeting and expense planning.")

        BONUS FEATURE 1 : DAY-OF-WEEK SPENDING ANALYSIS

Day-wise Spending Summary



,Day,Transactions,Total Spending,Average Spending
1,Monday,30,18326.0,610.866667
5,Tuesday,39,42390.0,1086.923077
6,Wednesday,33,39565.0,1198.939394
4,Thursday,39,15758.0,404.051282
0,Friday,41,19509.0,475.829268
2,Saturday,41,73929.0,1803.146341
3,Sunday,38,25151.0,661.868421



Highest Spending Day



,2
Day,Saturday
Transactions,41
Total Spending,73929.0
Average Spending,1803.146341



Lowest Spending Day



...


Top Vendor Spending by Day



,Day_Name,Vendor,Amount
54,Saturday,Amazon In,18273.0
61,Saturday,Imps Zerodha Coin,15000.0
80,Saturday,Upi Zerodha Coin@Axis,15000.0
154,Tuesday,Imps Zerodha Coin,15000.0
195,Wednesday,Upi Flipkart@Hdfcbank,14416.0
64,Saturday,Myntra Designs,11391.0
90,Sunday,Atm Wdl Icici,5000.0
146,Tuesday,Amazonin Marketplace,4485.0
192,Wednesday,Upi Amazonpay@Hdfcbank,4208.0
147,Tuesday,Amzn Intpymt,4062.0



Category-wise Spending by Day



,Day_Name,Category,Amount
0,Friday,Bills,565.0
1,Friday,Entertainment,453.0
2,Friday,Food,2609.0
3,Friday,Groceries,463.0
4,Friday,Others,7281.0
5,Friday,Shopping,2209.0
6,Friday,Transfer,3114.0
7,Friday,Travel,2815.0
8,Monday,Food,5528.0
9,Monday,Groceries,323.0



INTERPRETATION
• The highest spending occurred on Saturday.
• The lowest spending occurred on Thursday.
• Day-wise analysis helps identify weekly spending patterns.
• Vendor-wise and category-wise summaries reveal where money is spent most on different days.
• These insights support better weekly budgeting and expense planning.


In [16]:
# BONUS FEATURE 2: VENDOR CLEANUP AUDIT

print("=" * 70)
print("              BONUS FEATURE 2 : VENDOR CLEANUP AUDIT")
print("=" * 70)

vendor_audit = clean_df.copy()
vendor_audit["Clean_Vendor"] = (
    vendor_audit["Vendor"]
    .astype(str)
    .str.strip()
    .str.title()
)
vendor_mapping = {
    "Amazon Pay": "Amazon",
    "Amazon Seller": "Amazon",
    "Amazon Shopping": "Amazon",
    "Flipkart Internet": "Flipkart",
    "Flipkart Pvt Ltd": "Flipkart",
    "Swiggy Ltd": "Swiggy",
    "Swiggy Instamart": "Swiggy",
    "Zomato Limited": "Zomato",
    "Uber India": "Uber",
    "Ola Cabs": "Ola",
    "Reliance Fresh": "Reliance",
    "Reliance Retail": "Reliance",
    "Indian Oil": "Indian Oil",
    "Hpcl": "HPCL",
    "Bharat Petroleum": "BPCL"
}
vendor_audit["Clean_Vendor"] = (
    vendor_audit["Clean_Vendor"]
    .replace(vendor_mapping)
)
unknown_vendors = vendor_audit[
    vendor_audit["Clean_Vendor"] == "Unknown"
]
print("\nUnknown Vendor Transactions :", len(unknown_vendors))

before_count = (
    clean_df["Vendor"]
    .value_counts()
    .reset_index()
)
before_count.columns = ["Vendor", "Transactions"]
print("\nTop Vendors Before Cleaning\n")
display(before_count.head(10))
after_count = (
    vendor_audit["Clean_Vendor"]
    .value_counts()
    .reset_index()
)
after_count.columns = ["Vendor", "Transactions"]
print("\nTop Vendors After Cleaning\n")
display(after_count.head(10))
before_unique = clean_df["Vendor"].nunique()
after_unique = vendor_audit["Clean_Vendor"].nunique()
print("\nUnique Vendors Before Cleaning :", before_unique)
print("Unique Vendors After Cleaning  :", after_unique)
print("\nSample Vendor Cleanup\n")

display(
    vendor_audit[
        [
            "Description",
            "Vendor",
            "Clean_Vendor"
        ]
    ].head(20)
)
clean_df["Vendor"] = vendor_audit["Clean_Vendor"]

print("\n" + "=" * 70)
print("INTERPRETATION")
print("=" * 70)
print("• Vendor names were standardized by removing inconsistent spellings.")
print("• Similar merchant names were merged into a single vendor.")
print("• Unknown vendors were identified for manual verification.")
print("• Duplicate vendor representations were eliminated.")
print("• The cleaned vendor list improves the accuracy of vendor-wise spending analysis and reporting.")

              BONUS FEATURE 2 : VENDOR CLEANUP AUDIT

Unknown Vendor Transactions : 0

Top Vendors Before Cleaning



,Vendor,Transactions
0,Upi Rapido@Okaxis,17
1,Swiggy Instamart,16
2,Pos Swiggy Bangalore,14
3,Zomato Media P L,13
4,Pos Zepto Bangalore,12
5,Uber India,12
6,Upi Swiggy@Okaxis,12
7,Ola Electric,11
8,Ola Cabs,11
9,Upi Starbucks@Axis,11



Top Vendors After Cleaning



,Vendor,Transactions
0,Upi Rapido@Okaxis,17
1,Swiggy,16
2,Pos Swiggy Bangalore,14
3,Zomato Media P L,13
4,Pos Zepto Bangalore,12
5,Uber,12
6,Upi Swiggy@Okaxis,12
7,Ola Electric,11
8,Ola,11
9,Upi Starbucks@Axis,11



Unique Vendors Before Cleaning : 113
Unique Vendors After Cleaning  : 113

Sample Vendor Cleanup



,Description,Vendor,Clean_Vendor
1,BHIM-BMTC,Bhim Bmtc,Bhim Bmtc
4,BHIM-BLINKIT,Bhim Blinkit,Bhim Blinkit
9,UPI-GROWWPAY@HDFCBANK,Upi Growwpay@Hdfcbank,Upi Growwpay@Hdfcbank
12,POS OLA-PRIME,Pos Ola Prime,Pos Ola Prime
14,UPI-STARBUCKS@AXIS,Upi Starbucks@Axis,Upi Starbucks@Axis
16,ANI Technologies,Ani Technologies,Ani Technologies
21,GROFERS INDIA P L,Grofers India P L,Grofers India P L
24,BANGALORE ELEC SUPPLY,Bangalore Elec Supply,Bangalore Elec Supply
30,OLA CABS,Ola Cabs,Ola
32,UPI-AMAZONPAY@HDFCBANK,Upi Amazonpay@Hdfcbank,Upi Amazonpay@Hdfcbank



INTERPRETATION
• Vendor names were standardized by removing inconsistent spellings.
• Similar merchant names were merged into a single vendor.
• Unknown vendors were identified for manual verification.
• Duplicate vendor representations were eliminated.
• The cleaned vendor list improves the accuracy of vendor-wise spending analysis and reporting.


In [20]:
# BONUS FEATURE 3: NINTH SPENDING ARCHETYPE

print("=" * 70)
print("         BONUS FEATURE 3 : NINTH SPENDING ARCHETYPE")
print("=" * 70)

expense_df = clean_df[clean_df["Type"].str.lower() == "debit"].copy()

total_transactions = len(expense_df)
average_amount = expense_df["Amount"].mean()
small_transactions = expense_df[
    expense_df["Amount"] < average_amount
]
small_count = len(small_transactions)
small_percentage = (small_count / total_transactions) * 100
if small_percentage >= 60:
    ninth_archetype = "Impulse Buyer"
    description = (
        "The user frequently makes small-value purchases, "
        "indicating spontaneous spending behaviour."
    )
else:
    ninth_archetype = "Planned Spender"
    description = (
        "The user's purchases are generally planned and "
        "less impulsive."
    )

print("\nNinth Spending Archetype")
print("-" * 40)
print(f"Archetype               : {ninth_archetype}")
print(f"Average Expense         : ₹ {average_amount:.2f}")
print(f"Small Transactions      : {small_count}")
print(f"Total Transactions      : {total_transactions}")
print(f"Small Purchase %        : {small_percentage:.2f}%")
print("\nSample Small Transactions\n")

display(
    small_transactions[
        [
            "Date",
            "Vendor",
            "Category",
            "Amount"
        ]
    ].head(15)
)

print("\n" + "=" * 70)
print("INTERPRETATION")
print("=" * 70)
print(f"Detected Archetype : {ninth_archetype}")
print(description)
print("• The classification is based on the proportion of small-value transactions.")
print("• This rule-based approach does not use Machine Learning.")
print("• The archetype provides additional insight into the user's spending habits.")

         BONUS FEATURE 3 : NINTH SPENDING ARCHETYPE

Ninth Spending Archetype
----------------------------------------
Archetype               : Impulse Buyer
Average Expense         : ₹ 898.96
Small Transactions      : 227
Total Transactions      : 261
Small Purchase %        : 86.97%

Sample Small Transactions



,Date,Vendor,Category,Amount
4,2024-01-01,Bhim Blinkit,Others,270.0
14,2024-01-02,Upi Starbucks@Axis,Transfer,266.0
16,2024-01-02,Ani Technologies,Others,267.0
21,2024-03-01,Grofers India P L,Others,878.0
30,2024-04-01,Ola,Travel,284.0
33,2024-01-05,Pos Blinkit,Others,502.0
44,2024-06-01,Upi Zomato Limited@Paytm,Food,267.0
46,2024-01-06,Grofers India P L,Others,753.0
51,2024-01-07,Bhim Swiggy,Food,442.0
54,2024-01-07,Ola,Travel,351.0



INTERPRETATION
Detected Archetype : Impulse Buyer
The user frequently makes small-value purchases, indicating spontaneous spending behaviour.
• The classification is based on the proportion of small-value transactions.
• This rule-based approach does not use Machine Learning.
• The archetype provides additional insight into the user's spending habits.


In [22]:
# BONUS FEATURE 4: SPEND FORECASTING USING NUMPY

print("=" * 70)
print("          BONUS FEATURE 4 : SPEND FORECASTING")
print("=" * 70)

expense_df = clean_df[clean_df["Type"].str.lower() == "debit"].copy()
monthly_expense = (
    expense_df.groupby(["Year", "Month"])["Amount"]
    .sum()
    .reset_index()
)
monthly_expense = monthly_expense.sort_values(
    by=["Year", "Month"]
)
print("\nMonthly Expenses\n")
display(monthly_expense)

months = np.arange(1, len(monthly_expense) + 1)
expenses = monthly_expense["Amount"].values
if len(expenses) >= 2:
    slope, intercept = np.polyfit(months, expenses, 1)
    next_month = len(months) + 1
    predicted_expense = slope * next_month + intercept
else:
    slope = 0
    intercept = expenses[0] if len(expenses) == 1 else 0
    predicted_expense = intercept
print("\nLinear Trend Equation")
print(f"Expense = ({slope:.2f} × Month) + ({intercept:.2f})")
print("\nForecast for Next Month")
print(f"Predicted Expense : ₹ {predicted_expense:.2f}")
if len(expenses) >= 2:
    growth_rate = (
        (expenses[-1] - expenses[-2]) /
        expenses[-2]
    ) * 100
    print(f"Last Month Growth Rate : {growth_rate:.2f}%")
else:
    print("Growth Rate : Not enough data.")

comparison = monthly_expense.copy()
comparison["Month Number"] = months
display(comparison)

print("\n" + "=" * 70)
print("INTERPRETATION")
print("=" * 70)

print("• Monthly expenses were used to build a simple forecasting model.")
print("• NumPy's polyfit() function was used to estimate a linear trend.")
print(f"• Predicted expense for the next month is ₹{predicted_expense:.2f}.")
print("• The forecast is based on historical spending behaviour.")
print("• This simple model can help users plan future budgets.")

          BONUS FEATURE 4 : SPEND FORECASTING

Monthly Expenses



,Year,Month,Amount
0,2024,1,19296.0
1,2024,2,21480.0
2,2024,3,38299.0
3,2024,4,34345.0
4,2024,5,88051.0
5,2024,6,21979.0
6,2024,7,886.0
7,2024,8,640.0
8,2024,9,3664.0
9,2024,10,5503.0



Linear Trend Equation
Expense = (-3785.85 × Month) + (44044.89)

Forecast for Next Month
Predicted Expense : ₹ -1385.25
Last Month Growth Rate : -91.19%


,Year,Month,Amount,Month Number
0,2024,1,19296.0,1
1,2024,2,21480.0,2
2,2024,3,38299.0,3
3,2024,4,34345.0,4
4,2024,5,88051.0,5
5,2024,6,21979.0,6
6,2024,7,886.0,7
7,2024,8,640.0,8
8,2024,9,3664.0,9
9,2024,10,5503.0,10



INTERPRETATION
• Monthly expenses were used to build a simple forecasting model.
• NumPy's polyfit() function was used to estimate a linear trend.
• Predicted expense for the next month is ₹-1385.25.
• The forecast is based on historical spending behaviour.
• This simple model can help users plan future budgets.


In [23]:
# FINAL EXPENSE INTELLIGENCE REPORT

print("\n")
print("=" * 80)
print("               PERSONAL EXPENSE INTELLIGENCE REPORT")
print("=" * 80)

expense_df = clean_df[clean_df["Type"].str.lower() == "debit"]
income_df = clean_df[clean_df["Type"].str.lower() == "credit"]
total_income = income_df["Amount"].sum()
total_expense = expense_df["Amount"].sum()
net_savings = total_income - total_expense

print(f"Total Transactions      : {len(clean_df)}")
print(f"Total Income            : ₹ {total_income:.2f}")
print(f"Total Expenses          : ₹ {total_expense:.2f}")
print(f"Net Savings             : ₹ {net_savings:.2f}")
print("-" * 80)

top_category = (
    expense_df.groupby("Category")["Amount"]
    .sum()
    .sort_values(ascending=False)
)
print("Highest Spending Category")
print(f"Category : {top_category.index[0]}")
print(f"Amount   : ₹ {top_category.iloc[0]:.2f}")
print("-" * 80)

top_vendor = (
    expense_df.groupby("Vendor")["Amount"]
    .sum()
    .sort_values(ascending=False)
)
print("Top Vendor")
print(f"Vendor : {top_vendor.index[0]}")
print(f"Spent  : ₹ {top_vendor.iloc[0]:.2f}")
print("-" * 80)

monthly_expense = (
    expense_df.groupby(["Year", "Month_Name"])["Amount"]
    .sum()
    .reset_index()
)
highest_month = monthly_expense.loc[
    monthly_expense["Amount"].idxmax()
]
print("Highest Spending Month")
print(f"Month : {highest_month['Month_Name']}")
print(f"Spent : ₹ {highest_month['Amount']:.2f}")
print("-" * 80)
time_summary = (
    expense_df.groupby("Time_Slot")["Amount"]
    .sum()
    .sort_values(ascending=False)
)
print("Highest Spending Time")
print(f"{time_summary.index[0]} (₹ {time_summary.iloc[0]:.2f})")
print("-" * 80)
try:
    print(f"Total Anomalies Detected : {len(anomalies)}")
except:
    print("Total Anomalies Detected : Not Available")
print("-" * 80)
print("Detected Spending Archetypes")
try:
    if len(archetypes) == 0:
        print("Balanced Spender")
    else:
        for item in archetypes:
            print(f"• {item}")
except:
    print("No archetypes available.")
print("-" * 80)
try:
    print(f"Predicted Next Month Expense : ₹ {predicted_expense:.2f}")
except:
    print("Forecast Not Available")

print("=" * 80)
print("                  END OF REPORT")
print("=" * 80)



               PERSONAL EXPENSE INTELLIGENCE REPORT
Total Transactions      : 509
Total Income            : ₹ 0.00
Total Expenses          : ₹ 234628.00
Net Savings             : ₹ -234628.00
--------------------------------------------------------------------------------
Highest Spending Category
Category : Shopping
Amount   : ₹ 71928.00
--------------------------------------------------------------------------------
Top Vendor
Vendor : Imps Zerodha Coin
Spent  : ₹ 30000.00
--------------------------------------------------------------------------------
Highest Spending Month
Month : May
Spent : ₹ 88051.00
--------------------------------------------------------------------------------
Highest Spending Time
Night (₹ 234628.00)
--------------------------------------------------------------------------------
Total Anomalies Detected : 12
--------------------------------------------------------------------------------
Detected Spending Archetypes
• Shopaholic
--------------------------